# Initialize Spark

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("ITR1_amritap1")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.executor.cores", "2")
    .config("spark.executor.instances", "2")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/17 20:52:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Load data

In [2]:
commentsDF = spark.read.json("/home/amritap1/30123/final/comments")

In [3]:
submissionsDF = spark.read.json("/home/amritap1/30123/final/submissions")

25/10/17 20:52:54 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [4]:
commentsDF.printSchema()

root
 |-- approved_at_utc: string (nullable = true)
 |-- approved_by: string (nullable = true)
 |-- archived: boolean (nullable = true)
 |-- author: string (nullable = true)
 |-- author_cakeday: boolean (nullable = true)
 |-- author_created_utc: long (nullable = true)
 |-- author_flair_background_color: string (nullable = true)
 |-- author_flair_css_class: string (nullable = true)
 |-- author_flair_richtext: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- a: string (nullable = true)
 |    |    |-- e: string (nullable = true)
 |    |    |-- t: string (nullable = true)
 |    |    |-- u: string (nullable = true)
 |-- author_flair_template_id: string (nullable = true)
 |-- author_flair_text: string (nullable = true)
 |-- author_flair_text_color: string (nullable = true)
 |-- author_flair_type: string (nullable = true)
 |-- author_fullname: string (nullable = true)
 |-- author_patreon_flair: boolean (nullable = true)
 |-- banned_at_utc: string (nullabl

In [5]:
submissionsDF.printSchema()

root
 |-- approved_at_utc: string (nullable = true)
 |-- approved_by: string (nullable = true)
 |-- archived: boolean (nullable = true)
 |-- author: string (nullable = true)
 |-- author_cakeday: boolean (nullable = true)
 |-- author_created_utc: long (nullable = true)
 |-- author_flair_background_color: string (nullable = true)
 |-- author_flair_css_class: string (nullable = true)
 |-- author_flair_richtext: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- a: string (nullable = true)
 |    |    |-- e: string (nullable = true)
 |    |    |-- t: string (nullable = true)
 |    |    |-- u: string (nullable = true)
 |-- author_flair_template_id: string (nullable = true)
 |-- author_flair_text: string (nullable = true)
 |-- author_flair_text_color: string (nullable = true)
 |-- author_flair_type: string (nullable = true)
 |-- author_fullname: string (nullable = true)
 |-- author_patreon_flair: boolean (nullable = true)
 |-- banned_at_utc: string (nullabl

# Clean Data

In [6]:
from pyspark.sql import functions as F

def clean_df_spark(df, columns_to_keep, text_column):
    df = df.select(*columns_to_keep)

    # Convert timestamps (epoch seconds) to timestamp
    if "created_utc" in df.columns:
        df = df.withColumn(
            "created_utc",
            F.to_timestamp(F.from_unixtime(F.col("created_utc").cast("bigint")))
        )

    # Remove deleted/empty content
    if text_column in df.columns:
        df = df.filter(
            F.col(text_column).isNotNull() &
            ~F.col(text_column).isin("[deleted]", "[removed]")
        )

    # Remove bot accounts
    if "author" in df.columns:
        a = F.lower(F.col("author"))
        is_bot = F.coalesce(
            a.contains("automoderator") | a.contains("moderator") | a.contains("bot") | a.contains("automod"),
            F.lit(False)
        )
        df = df.filter(~is_bot)

    return df

In [7]:
commentsDF_clean = clean_df_spark(
    commentsDF,
    ['id', 'body', 'score', 'created_utc', 'author'],
    'body'
)

In [8]:
submissionsDF_clean = clean_df_spark(
    submissionsDF,
    ['id', 'title', 'selftext', 'score', 'num_comments', 'created_utc', 'author'],
    'selftext'
)

In [9]:
commentsDF_clean.printSchema()

root
 |-- id: string (nullable = true)
 |-- body: string (nullable = true)
 |-- score: long (nullable = true)
 |-- created_utc: timestamp (nullable = true)
 |-- author: string (nullable = true)



In [10]:
submissionsDF_clean.printSchema()

root
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- selftext: string (nullable = true)
 |-- score: long (nullable = true)
 |-- num_comments: long (nullable = true)
 |-- created_utc: timestamp (nullable = true)
 |-- author: string (nullable = true)



In [11]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, NGram, CountVectorizer, MinHashLSH

In [ ]:
# Normalize to (id, src, text)
comments = commentsDF_clean.select(
    F.col("id").alias("id"),
    F.lit("comment").alias("src"),
    F.col("body").alias("text")
).where(F.col("text").isNotNull())

subs = submissionsDF_clean.select(
    F.col("id").alias("id"),
    F.lit("submission").alias("src"),
    F.concat_ws(" ", F.coalesce("title", F.lit("")), F.coalesce("selftext", F.lit(""))).alias("text")
).where(F.col("text").isNotNull())

both = comments.unionByName(subs)

In [13]:
comments.show()

+-------+-------+--------------------+
|     id|    src|                text|
+-------+-------+--------------------+
|e1ksyul|comment|                 Lol|
|e1kt1dp|comment|I was saying that...|
|e1kt1r4|comment|Well that doesn't...|
|e1kt2lq|comment|Buy bull call spr...|
|e1kt2xf|comment|Seriously? Sounds...|
|e1kt5ak|comment|Why the fuck is e...|
|e1kt5fi|comment|And make sure to ...|
|e1kt5jh|comment|Low for the day w...|
|e1kt6bz|comment|What's the most f...|
|e1ktade|comment|You can have repr...|
|e1ktbbq|comment|[Macabilly IRL](h...|
|e1ktbpw|comment|Assuming that you...|
|e1ktbyy|comment|I'm pretty strong...|
|e1ktdav|comment|       u/theycallme1|
|e1ktde2|comment|"The state of nat...|
|e1ktfxp|comment|I think immigrant...|
|e1ktgkz|comment|So you mean I can...|
|e1kthaw|comment|Fascinating stuff...|
|e1kthku|comment|Your argument tha...|
|e1ktjbk|comment|fishbum30 for pre...|
+-------+-------+--------------------+
only showing top 20 rows



In [14]:
subs.show()

+------+----------+--------------------+
|    id|       src|                text|
+------+----------+--------------------+
|93jwhw|submission|Do I get anything...|
|93jwnw|submission|$IQ Earnings Call...|
|93jwqw|submission|Anyone else on th...|
|93k17v|submission|Baidu: Beat Earni...|
|93k1ug|submission|E*TRADE day tradi...|
|93k2lz|submission|Calls on companie...|
|93k5nu|submission|Is this how you Y...|
|93k7jj|submission|Found the godfath...|
|93k8f8|submission|YO PUT THIS SHIT!...|
|93ka72|submission|WSBers need love ...|
|93kaex|submission|Fed seen keeping ...|
|93kbiw|submission|Cursing is down r...|
|93ke9u|submission|My FD for $PWRB t...|
|93kjbl|submission|shitpost before T...|
|93knyu|submission|Hey guys I figure...|
|93koll|submission|$MSFT $102 put 8/...|
|93ktq3|submission|StockTwats making...|
|93kw9c|submission|Baidu Sales, Prof...|
|93kzng|submission|SQ ER Give me you...|
|93l11f|submission|Wafer. Fabricatio...|
+------+----------+--------------------+
only showing top

# Locality Sensitive Hashing

## Jaccard Distance - Set Similarity

### Using binary count vectorizer

In [15]:
# MinHash LSH (Jaccard over word ngrams)
tok = RegexTokenizer(inputCol="text", outputCol="tokens", pattern="\\W+")
sw  = StopWordsRemover(inputCol="tokens", outputCol="clean")
ng  = NGram(n=3, inputCol="clean", outputCol="ngrams")
cv  = CountVectorizer(inputCol="ngrams", outputCol="features", binary=True, minDF=2)

In [16]:
# Average no of ngrams (n=3) per src
pre = Pipeline(stages=[tok, sw, ng]).fit(both).transform(both)
pre.groupBy("src").agg(F.avg(F.size("ngrams")).alias("avg_ngrams")).show()

[Stage 6:=====================================================>   (28 + 2) / 30]

+----------+-----------------+
|       src|       avg_ngrams|
+----------+-----------------+
|   comment|8.840000740003184|
|submission|21.55629374679397|
+----------+-----------------+



In [17]:
from pyspark.sql.types import IntegerType

nnz_udf = F.udf(lambda v: int(v.numNonzeros()) if v else 0, IntegerType())

featurizer = Pipeline(stages=[tok, sw, ng, cv]).fit(both)
feats = featurizer.transform(both).withColumn("nnz", nnz_udf("features"))

In [18]:
from pyspark.sql import functions as F
from pyspark.sql.types import BooleanType
from pyspark.ml.feature import MinHashLSH

has_vec = F.udf(lambda v: (v is not None) and (getattr(v, "size", 0) > 0), BooleanType())
has_nz  = F.udf(lambda v: (v is not None) and (getattr(v, "numNonzeros", lambda:0)() > 0), BooleanType())
feats_clean = (feats
               .where(has_vec("features") & has_nz("features"))
               .select("id","src","features")
               .cache())
feats_clean.count()

25/10/17 00:11:44 WARN DAGScheduler: Broadcasting large task binary with size 4.8 MiB
25/10/17 00:12:55 WARN DAGScheduler: Broadcasting large task binary with size 4.8 MiB
                                                                                

1252691

In [19]:
# Split only from the clean frame
f_comm = feats_clean.where(F.col("src")=="comment").select("id","features")
f_subs = feats_clean.where(F.col("src")=="submission").select("id","features")

assert f_comm.where(~has_nz("features")).count() == 0
assert f_subs.where(~has_nz("features")).count() == 0

25/10/17 00:12:57 WARN DAGScheduler: Broadcasting large task binary with size 4.8 MiB
25/10/17 00:13:07 WARN DAGScheduler: Broadcasting large task binary with size 4.8 MiB
                                                                                

In [20]:
lsh = MinHashLSH(inputCol="features", outputCol="hashes", numHashTables=16).fit(feats_clean)

In [21]:
ac = lsh.transform(f_comm)
ac.count()

25/10/17 00:13:10 WARN DAGScheduler: Broadcasting large task binary with size 4.8 MiB
                                                                                

1193034

In [22]:
bs = lsh.transform(f_subs)
bs.count()

25/10/17 00:13:12 WARN DAGScheduler: Broadcasting large task binary with size 4.8 MiB
                                                                                

59657

In [48]:
from pyspark.sql import functions as F
from pyspark.sql.types import BooleanType
from pyspark.ml.feature import MinHashLSH

has_vec = F.udf(lambda v: (v is not None) and (getattr(v, "size", 0) > 0), BooleanType())
has_nz  = F.udf(lambda v: (v is not None) and (getattr(v, "numNonzeros", lambda:0)() > 0), BooleanType())
feats_clean = (feats
               .where(has_vec("features") & has_nz("features"))
               .select("id","src","features")
               .cache())
feats_clean.count()

25/10/17 17:58:29 WARN DAGScheduler: Broadcasting large task binary with size 4.8 MiB
25/10/17 17:59:47 WARN DAGScheduler: Broadcasting large task binary with size 4.8 MiB
                                                                                

1256145

In [23]:
pairs = (lsh.approxSimilarityJoin(ac, bs, 0.7, "jaccardDist")
         .select(F.col("datasetA.id").alias("comment_id"),
                 F.col("datasetB.id").alias("submission_id"),
                 "jaccardDist")
         .orderBy("jaccardDist"))

In [24]:
pairs.groupBy("jaccardDist").count().orderBy("jaccardDist").show()

25/10/17 00:13:14 WARN DAGScheduler: Broadcasting large task binary with size 4.8 MiB
25/10/17 00:13:14 WARN DAGScheduler: Broadcasting large task binary with size 4.8 MiB
25/10/17 00:13:44 WARN DAGScheduler: Broadcasting large task binary with size 4.9 MiB
25/10/17 00:19:08 WARN TaskMemoryManager: Failed to allocate a page (16777200 bytes), try again.
25/10/17 00:19:09 WARN TaskMemoryManager: Failed to allocate a page (16777200 bytes), try again.
25/10/17 00:19:09 WARN TaskMemoryManager: Failed to allocate a page (16777200 bytes), try again.
25/10/17 00:19:12 WARN TaskMemoryManager: Failed to allocate a page (16777200 bytes), try again.


[588.401s][warning][gc,alloc] Executor task launch worker for task 50.0 in stage 33.0 (TID 568): Retried waiting for GCLocker too often allocating 2097152 words


25/10/17 00:19:15 WARN TaskMemoryManager: Failed to allocate a page (16777200 bytes), try again.
25/10/17 00:20:43 WARN DAGScheduler: Broadcasting large task binary with size 4.9 MiB
[Stage 37:=====================================================>(199 + 1) / 200]

+--------------------+-------+
|         jaccardDist|  count|
+--------------------+-------+
|                 0.0|1093521|
|0.021276595744680882|      1|
|0.023255813953488413|      1|
|0.025000000000000022|      1|
| 0.02564102564102566|      1|
| 0.03418803418803418|      2|
|  0.0357142857142857|      1|
| 0.04166666666666663|      1|
| 0.04761904761904767|      1|
|0.050000000000000044|      1|
|0.052631578947368474|      1|
| 0.05405405405405406|      1|
| 0.05555555555555558|      1|
|0.061224489795918324|      1|
|              0.0625|      1|
| 0.06451612903225812|      1|
| 0.06666666666666665|      1|
|  0.0714285714285714|      7|
| 0.07377049180327866|      2|
| 0.07692307692307687|      7|
+--------------------+-------+
only showing top 20 rows



25/10/17 00:22:26 WARN DAGScheduler: Broadcasting large task binary with size 4.8 MiB
                                                                                

In [25]:
from pyspark.ml.feature import Bucketizer
from pyspark.sql import functions as F

splits = [-float("inf"), 0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, float("inf")]
bkt = Bucketizer(splits=splits, inputCol="jaccardDist", outputCol="bin_id", handleInvalid="keep")
binned = bkt.transform(pairs)

labels = [f"[{splits[i]},{splits[i+1]})" for i in range(len(splits)-1)]
label_map = F.create_map(*sum([[F.lit(i), F.lit(lbl)] for i,lbl in enumerate(labels)], []))

out = (binned.withColumn("bucket", label_map[F.col("bin_id")])
              .groupBy("bucket").count()
              .orderBy("bucket"))
out.show(truncate=False)

25/10/17 00:25:24 WARN DAGScheduler: Broadcasting large task binary with size 4.8 MiB
25/10/17 00:25:24 WARN DAGScheduler: Broadcasting large task binary with size 4.8 MiB
25/10/17 00:25:55 WARN DAGScheduler: Broadcasting large task binary with size 4.9 MiB
[Stage 47:=======================================>                (47 + 4) / 67]

[1314.015s][warning][gc,alloc] Executor task launch worker for task 50.0 in stage 47.0 (TID 896): Retried waiting for GCLocker too often allocating 2097154 words


25/10/17 00:31:21 WARN TaskMemoryManager: Failed to allocate a page (16777216 bytes), try again.
25/10/17 00:31:21 WARN TaskMemoryManager: Failed to allocate a page (16777216 bytes), try again.
25/10/17 00:31:21 WARN TaskMemoryManager: Failed to allocate a page (16777216 bytes), try again.
25/10/17 00:31:22 WARN TaskMemoryManager: Failed to allocate a page (16777216 bytes), try again.
25/10/17 00:31:22 WARN TaskMemoryManager: Failed to allocate a page (16777216 bytes), try again.
25/10/17 00:31:22 WARN TaskMemoryManager: Failed to allocate a page (16777216 bytes), try again.
25/10/17 00:31:23 WARN TaskMemoryManager: Failed to allocate a page (16777216 bytes), try again.
25/10/17 00:31:23 WARN TaskMemoryManager: Failed to allocate a page (16777216 bytes), try again.


[1316.186s][warning][gc,alloc] Executor task launch worker for task 50.0 in stage 47.0 (TID 896): Retried waiting for GCLocker too often allocating 2097154 words


25/10/17 00:32:56 WARN DAGScheduler: Broadcasting large task binary with size 4.9 MiB
25/10/17 00:34:44 WARN DAGScheduler: Broadcasting large task binary with size 4.8 MiB


+---------+-------+
|bucket   |count  |
+---------+-------+
|[0.0,0.1)|1093578|
|[0.1,0.2)|222169 |
|[0.2,0.3)|174345 |
|[0.3,0.4)|475811 |
|[0.4,0.5)|481837 |
|[0.5,0.6)|3978521|
|[0.6,0.7)|5602702|
+---------+-------+



### Using frequencies in count vectorizer

In [53]:
# MinHash LSH (Jaccard over word ngrams)
tok = RegexTokenizer(inputCol="text", outputCol="tokens", pattern="\\W+")
sw  = StopWordsRemover(inputCol="tokens", outputCol="clean")
ng  = NGram(n=3, inputCol="clean", outputCol="ngrams")
cv  = CountVectorizer(inputCol="ngrams", outputCol="features", minDF=5)

In [54]:
from pyspark.sql.types import IntegerType

nnz_udf = F.udf(lambda v: int(v.numNonzeros()) if v else 0, IntegerType())

featurizer = Pipeline(stages=[tok, sw, ng, cv]).fit(both)
feats = featurizer.transform(both).withColumn("nnz", nnz_udf("features"))

In [55]:
from pyspark.sql import functions as F
from pyspark.sql.types import BooleanType
from pyspark.ml.feature import MinHashLSH

has_vec = F.udf(lambda v: (v is not None) and (getattr(v, "size", 0) > 0), BooleanType())
has_nz  = F.udf(lambda v: (v is not None) and (getattr(v, "numNonzeros", lambda:0)() > 0), BooleanType())
feats_clean = (feats
               .where(has_vec("features") & has_nz("features"))
               .select("id","src","features")
               .cache())
feats_clean.count()

25/10/17 18:01:25 WARN DAGScheduler: Broadcasting large task binary with size 4.8 MiB
25/10/17 18:02:40 WARN DAGScheduler: Broadcasting large task binary with size 4.8 MiB
                                                                                

1256184

In [56]:
# Split only from the clean frame
f_comm = feats_clean.where(F.col("src")=="comment").select("id","features")
f_subs = feats_clean.where(F.col("src")=="submission").select("id","features")

assert f_comm.where(~has_nz("features")).count() == 0
assert f_subs.where(~has_nz("features")).count() == 0

25/10/17 18:02:41 WARN DAGScheduler: Broadcasting large task binary with size 4.8 MiB
25/10/17 18:02:52 WARN DAGScheduler: Broadcasting large task binary with size 4.8 MiB
                                                                                

In [57]:
lsh = MinHashLSH(inputCol="features", outputCol="hashes", numHashTables=8).fit(feats_clean)

In [59]:
ac = lsh.transform(f_comm)
bs = lsh.transform(f_subs)

In [60]:
pairs = (lsh.approxSimilarityJoin(ac, bs, 0.6, "jaccardDist")
         .select(F.col("datasetA.id").alias("comment_id"),
                 F.col("datasetB.id").alias("submission_id"),
                 "jaccardDist")
         .orderBy("jaccardDist"))

In [61]:
pairs.groupBy("jaccardDist").count().orderBy("jaccardDist").show()

25/10/17 18:03:06 WARN DAGScheduler: Broadcasting large task binary with size 4.8 MiB
25/10/17 18:03:07 WARN DAGScheduler: Broadcasting large task binary with size 4.8 MiB
25/10/17 18:03:24 WARN DAGScheduler: Broadcasting large task binary with size 4.9 MiB
25/10/17 18:05:17 WARN DAGScheduler: Broadcasting large task binary with size 4.9 MiB
[Stage 121:=====================================================> (65 + 2) / 67]

+--------------------+-------+
|         jaccardDist|  count|
+--------------------+-------+
|                 0.0|1088535|
|0.019607843137254943|      1|
|0.023255813953488413|      1|
| 0.02564102564102566|      2|
| 0.03418803418803418|      2|
| 0.03703703703703709|      1|
| 0.04166666666666663|      1|
|  0.0423728813559322|      2|
|0.050000000000000044|      2|
| 0.05405405405405406|      1|
| 0.05555555555555558|      1|
| 0.05660377358490565|      1|
|              0.0625|      1|
| 0.06451612903225812|      1|
| 0.06666666666666665|      1|
|  0.0714285714285714|      7|
| 0.07627118644067798|      1|
| 0.07692307692307687|      6|
| 0.07894736842105265|      1|
| 0.08108108108108103|      1|
+--------------------+-------+
only showing top 20 rows



25/10/17 18:05:41 WARN DAGScheduler: Broadcasting large task binary with size 4.8 MiB
                                                                                

In [62]:
from pyspark.ml.feature import Bucketizer
from pyspark.sql import functions as F

splits = [-float("inf"), 0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, float("inf")]
bkt = Bucketizer(splits=splits, inputCol="jaccardDist", outputCol="bin_id", handleInvalid="keep")
binned = bkt.transform(pairs)

labels = [f"[{splits[i]},{splits[i+1]})" for i in range(len(splits)-1)]
label_map = F.create_map(*sum([[F.lit(i), F.lit(lbl)] for i,lbl in enumerate(labels)], []))

out = (binned.withColumn("bucket", label_map[F.col("bin_id")])
              .groupBy("bucket").count()
              .orderBy("bucket"))
out.show(truncate=False)

25/10/17 18:05:41 WARN DAGScheduler: Broadcasting large task binary with size 4.8 MiB
25/10/17 18:05:41 WARN DAGScheduler: Broadcasting large task binary with size 4.8 MiB
25/10/17 18:05:57 WARN DAGScheduler: Broadcasting large task binary with size 4.9 MiB
25/10/17 18:07:48 WARN DAGScheduler: Broadcasting large task binary with size 4.9 MiB
[Stage 135:======================================================>(66 + 1) / 67]

+---------+-------+
|bucket   |count  |
+---------+-------+
|[0.0,0.1)|1088591|
|[0.1,0.2)|222314 |
|[0.2,0.3)|170244 |
|[0.3,0.4)|484831 |
|[0.4,0.5)|483192 |
|[0.5,0.6)|3938155|
+---------+-------+



25/10/17 18:08:11 WARN DAGScheduler: Broadcasting large task binary with size 4.8 MiB
                                                                                

## Euclidean Distance - Vector Distance

In [16]:
from pyspark.sql import functions as F, Window
from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, HashingTF, IDF, Normalizer, BucketedRandomProjectionLSH

tf  = HashingTF(inputCol="clean", outputCol="tf", numFeatures=1<<18)
idf = IDF(inputCol="tf", outputCol="rawFeatures")
norm= Normalizer(inputCol="rawFeatures", outputCol="features", p=2.0)  # makes Euclid ~ cosine
brp = BucketedRandomProjectionLSH(inputCol="features", outputCol="buckets",
                                  bucketLength=1.0, numHashTables=16)

In [17]:
pipe  = Pipeline(stages=[tok, sw, tf, idf, norm, brp]).fit(both)
feats = pipe.transform(both).select("id","src","features").cache()
f_comm = feats.where("src='comment'").drop("src")
f_subs = feats.where("src='submission'").drop("src")

model = pipe.stages[-1]

25/10/17 20:53:38 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
25/10/17 20:53:38 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS


### Per-item ANN (top-k submissions for one comment)

In [ ]:
target_id = "e1kt2lq"
key_vec = f_comm.where(F.col("id")==target_id).select("features").head()[0]
topk_one = model.approxNearestNeighbors(f_subs, key_vec, 5) \
                .select(F.col("id").alias("submission_id"), F.col("distCol").alias("euclidDist"))

25/10/17 10:28:54 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB


In [23]:
topk_one.show()

25/10/17 10:29:10 WARN DAGScheduler: Broadcasting large task binary with size 36.1 MiB
[Stage 12:==============================================>         (25 + 4) / 30]

+-------------+------------------+
|submission_id|        euclidDist|
+-------------+------------------+
|       95f3ck|1.0000000000000002|
|       3csu9h|1.0000000000000002|
|       4shug1|1.0000000000000002|
|       3d4m4u|1.0000000000000002|
|       8lp20f|1.0000000000000002|
+-------------+------------------+



### Batch ANN: top-k submissions for every comment

In [ ]:
threshold = 2
SEED = 42
MAX_COMM = 1500
MAX_SUBS = 300

f_comm_s = f_comm.orderBy(F.rand(SEED)).limit(MAX_COMM).persist()
f_subs_s = f_subs.orderBy(F.rand(SEED)).limit(MAX_SUBS).persist()

In [19]:
f_comm_s.count()

25/10/17 20:53:39 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
25/10/17 20:54:19 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
25/10/17 20:54:20 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
25/10/17 20:54:20 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
                                                                                

1500

In [20]:
f_subs_s.count()

25/10/17 20:54:20 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
25/10/17 20:54:20 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
25/10/17 20:54:20 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB


300

In [ ]:
# Candidate generation (LSH join)
cand = (model.approxSimilarityJoin(f_comm_s, f_subs_s, threshold, distCol="euclidDist")
        .select(
            F.col("datasetA.id").alias("comment_id"),
            F.col("datasetB.id").alias("submission_id"),
            "euclidDist"
        ).persist())

In [22]:
cand.count()

25/10/17 20:54:21 WARN DAGScheduler: Broadcasting large task binary with size 36.2 MiB
25/10/17 20:54:21 WARN DAGScheduler: Broadcasting large task binary with size 36.3 MiB
25/10/17 20:54:35 WARN DAGScheduler: Broadcasting large task binary with size 36.3 MiB
                                                                                

449992

In [23]:
w = Window.partitionBy("comment_id").orderBy(F.col("euclidDist").asc(), F.col("submission_id").asc())
topk_all = (cand.withColumn("rk", F.row_number().over(w))
                 .where(F.col("rk") <= 5)
                 .drop("rk")
                 .withColumn("cosineSim", 1 - (F.col("euclidDist")**2)/F.lit(2.0)))

In [24]:
topk_all.show()

25/10/17 20:54:35 WARN DAGScheduler: Broadcasting large task binary with size 36.3 MiB


+----------+-------------+------------------+--------------------+
|comment_id|submission_id|        euclidDist|           cosineSim|
+----------+-------------+------------------+--------------------+
|   cf7eznv|       68ic44| 1.345945764441014| 0.09421499959164725|
|   cf7eznv|       24kf8t|1.3530669070077697| 0.08460497258021382|
|   cf7eznv|       1vdnn3|1.3562786744059654|  0.0802540786757987|
|   cf7eznv|       aaa2eu|1.3654192870073802| 0.06781508533412872|
|   cf7eznv|       3irqcs|1.3654618428860779| 0.06775697781107803|
|   cfe14k0|       52ka55|1.3492336421260323| 0.08978428947766093|
|   cfe14k0|       6sm3i0|1.3523589400271228| 0.08556264866435848|
|   cfe14k0|       49vmzo|1.3566727452670755|  0.0797195311247485|
|   cfe14k0|       6rksll|1.4142135623730947|4.440892098500626...|
|   cfe14k0|       1wosg9| 1.414213562373095|2.220446049250313...|
|   cfiygug|       72g9ny|  1.33559755090849| 0.10808959100362159|
|   cfiygug|       93zx5d|1.3545367713138976| 0.08261506757926

In [26]:
from pyspark.sql import functions as F, Window

w = Window.partitionBy("comment_id").orderBy(F.col("euclidDist").asc(), F.col("submission_id").asc())

topk_all = (
    cand.withColumn("rk", F.row_number().over(w))
        .where(F.col("rk") <= 5)
        .drop("rk")
        .withColumn("cosineSim", 1 - (F.col("euclidDist")**2) / F.lit(2.0))
        .withColumn("approx_euclidean_distance", (F.col("euclidDist") / 0.1).cast("int") * 0.1)
)

topk_all.select("comment_id", "submission_id", "euclidDist", "cosineSim", "approx_euclidean_distance").show(10)

topk_all.groupBy("approx_euclidean_distance").agg(
    F.count("*").alias("count"),
    F.avg("cosineSim").alias("avgCosineSim")
).orderBy("approx_euclidean_distance").show()

25/10/17 20:57:27 WARN DAGScheduler: Broadcasting large task binary with size 36.3 MiB


+----------+-------------+------------------+--------------------+-------------------------+
|comment_id|submission_id|        euclidDist|           cosineSim|approx_euclidean_distance|
+----------+-------------+------------------+--------------------+-------------------------+
|   cf7eznv|       68ic44| 1.345945764441014| 0.09421499959164725|                      1.3|
|   cf7eznv|       24kf8t|1.3530669070077697| 0.08460497258021382|                      1.3|
|   cf7eznv|       1vdnn3|1.3562786744059654|  0.0802540786757987|                      1.3|
|   cf7eznv|       aaa2eu|1.3654192870073802| 0.06781508533412872|                      1.3|
|   cf7eznv|       3irqcs|1.3654618428860779| 0.06775697781107803|                      1.3|
|   cfe14k0|       52ka55|1.3492336421260323| 0.08978428947766093|                      1.3|
|   cfe14k0|       6sm3i0|1.3523589400271228| 0.08556264866435848|                      1.3|
|   cfe14k0|       49vmzo|1.3566727452670755|  0.0797195311247485|    

25/10/17 20:57:28 WARN DAGScheduler: Broadcasting large task binary with size 36.3 MiB


+-------------------------+-----+--------------------+
|approx_euclidean_distance|count|        avgCosineSim|
+-------------------------+-----+--------------------+
|                      0.0|    2|                 1.0|
|       0.6000000000000001|    3|  0.7971220366695656|
|                      0.8|    4|   0.648797058256964|
|                      0.9|   97|  0.5061780695516584|
|                      1.0|   37| 0.43779537648916367|
|                      1.1|  138| 0.32644787687957255|
|       1.2000000000000002|  911| 0.19872073972768542|
|                      1.3| 5385| 0.08591456076281596|
|       1.4000000000000001|  923|9.267382234499514E-4|
+-------------------------+-----+--------------------+



# Experimentation Process

Used the r/wallstreetbets subreddit data (posts/submissions and comments) between 2014-2018 for analysis, and cleaned it to remove bot accounts and deleted rows.

## MinHash

Used MinHashLSH and NGram to work with Jaccard distances as a measure of set similarity - the similarity between tri-grams present in the reddit data.

Tri-gram analysis indicates 8.84 average counts for comments, and 21.56 for the submissions/posts.

### Binary Tri-grams

Used 16 hash tables and a join radius of 0.7 along with binary tri-grams and filtered out tokens not available in at least 2 data points.

After the data was cleaned for non-zero features vectors, around 1M pairs were returned with a jaccard distance of 0, indicating that the model received many exact tri-gram set duplicates, which makes sense in the context of binary tri-grams, as a lot of the same words are likely reused across submissions and comments.

Cleaning for non-zero features vector was challenging - there were frequent following errors in the pipeline:
> IllegalArgumentException: Must have at least 1 non zero entry.

indicating some empty feature vectors were being retained with earlier versions of the filter.

### Frequency Weighted Trigrams

Used frequency weighted tri-grams instead of binary tri-grams, and set the tokens being used to ones available in at least 5 documents.

The filter held 1,256,184 rows, and while there were still around 1M exact matches, around 3.9M pairs had a distance between 0.5 and 0.6, indicating moderate similarity.

Performance on the dataset was an issue, and further analysis with larger capacity clusters would be needed with higher threshold to fully explore the data

## Bucketed Random Projection on TF/IDF

Used BucketedRandomProjectionLSH with TF/IDF for measuring semantic distances, used euclidean distance of feature vectors normalised to unit length as the measure

### Per-item ANN

Extracted top-5 submissions similar to one comment - results were all at a distance of 1, indicating low degree of similarity between data points within the dataset.

The bucketing strategy would have to be more fine grained for this approach, but the similar nature of various data points would remain a challenge. Further cleaning of the data is also needed - likely for dropping data points with a large number of common tokens that might not be very meaningful without the context.

### Batch ANN

Took a sample of 1500 comments and 300 submissions, resulting in pairs with largely weak matches for the given samples similar to per-item ANN. Investigating the pairs with higher similarity scores might be helpful for the analysis.

Performance was a large issue for bucketed random projection with the used spark configuration, and I was unable to get a result for the entire dataset even with processing for hours on end. For a quick analysis, I used a limited subset of comments and submissions. An analysis with a larger capacity cluster and further performance tuning would be required to further investigate this data.